In [0]:
# Add widgets (parameters)
from pyspark.sql import functions as F
from datetime import datetime, timezone

dbutils.widgets.text("oct_path", "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")
dbutils.widgets.text("nov_path", "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")
dbutils.widgets.text("bronze_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events")
dbutils.widgets.text("batch_id", f"day07_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}")

oct_path = dbutils.widgets.get("oct_path")
nov_path = dbutils.widgets.get("nov_path")
bronze_path = dbutils.widgets.get("bronze_path")
batch_id = dbutils.widgets.get("batch_id")


In [0]:
raw = (
    spark.read.option("header", True).option("inferSchema", True).csv(oct_path)
    .unionByName(spark.read.option("header", True).option("inferSchema", True).csv(nov_path))
)

bronze = (
    raw
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingestion_ts", F.current_timestamp())
)

bronze.write.format("delta").mode("overwrite").save(bronze_path)

dbutils.notebook.exit(f"OK bronze batch_id={batch_id} rows={bronze.count()}")
